# Pipeline 3 - Next Best Channel Predictor

Generated: 2026-04-06T23:04:06.434746Z


## 1) Problem Framing

**Business question:** Recommend the next donation channel most likely to work for each supporter.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [ ]:
import json
import re
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, precision_recall_fscore_support, r2_score, roc_auc_score, top_k_accuracy_score
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "ml-pipelines" else cwd
DATA_DIR = REPO_ROOT / "data" / "raw"
OUT_DIR = REPO_ROOT / "output" / "ml-predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

def require_csv(stem):
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the INTEX CSVs under data/raw/.")
    return pd.read_csv(path, encoding="utf-8-sig")

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def as_bool(series):
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])

def numeric(series, fill_value=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(fill_value)

def fill_numeric_median(df, cols):
    for col in cols:
        vals = pd.to_numeric(df[col], errors="coerce")
        df[col] = vals.fillna(vals.median() if vals.notna().any() else 0.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None and pd.Series(y_true).nunique() > 1:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out

def eval_regression(y_true, y_pred):
    return {"mae": float(mean_absolute_error(y_true, y_pred)), "rmse": rmse(y_true, y_pred), "r2": float(r2_score(y_true, y_pred))}

def classification_baseline(y_train, y_test):
    majority = pd.Series(y_train).mode().iloc[0]
    pred = pd.Series([majority] * len(y_test), index=pd.Series(y_test).index)
    return {"majority_class": str(majority), "accuracy": float(accuracy_score(y_test, pred))}

def regression_baseline(y_train, y_test):
    median_value = float(pd.Series(y_train).median())
    pred = np.repeat(median_value, len(y_test))
    out = eval_regression(y_test, pred)
    out["baseline_value"] = median_value
    return out

def top_features(pipe, n=10):
    if "pre" not in pipe.named_steps:
        return pd.DataFrame()
    model = pipe.named_steps.get("model") or pipe.named_steps.get("m")
    if model is None:
        return pd.DataFrame()
    try:
        names = pipe.named_steps["pre"].get_feature_names_out()
    except Exception:
        return pd.DataFrame()
    if hasattr(model, "feature_importances_"):
        weights = np.asarray(model.feature_importances_)
    elif hasattr(model, "coef_"):
        weights = np.abs(np.asarray(model.coef_)).mean(axis=0) if np.asarray(model.coef_).ndim > 1 else np.abs(np.asarray(model.coef_))
    else:
        return pd.DataFrame()
    out = pd.DataFrame({"feature": names, "importance": weights}).sort_values("importance", ascending=False).head(n)
    return out.reset_index(drop=True)

def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)

def compact_cv_classification(X, y, cat_cols, num_cols, scoring_metric="roc_auc", cv_splits=3):
    y = pd.Series(y)
    min_class = y.value_counts().min()
    if y.nunique() < 2 or min_class < 2:
        print("Skipping CV comparison: target does not have enough samples in each class.")
        return pd.DataFrame()
    n_splits = int(min(cv_splits, min_class))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    models = {
        "Logistic Regression": LogisticRegression(max_iter=3000),
        "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42, min_samples_leaf=3),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    }
    rows = []
    scoring = {"accuracy": "accuracy", "f1_weighted": "f1_weighted"}
    if y.nunique() == 2:
        scoring["roc_auc"] = "roc_auc"
    for name, model in models.items():
        pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", model)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=1, error_score=np.nan)
        row = {"Model": name}
        for metric in scoring:
            vals = scores[f"test_{metric}"]
            row[f"{metric}_mean"] = float(np.nanmean(vals))
            row[f"{metric}_std"] = float(np.nanstd(vals))
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(f"{'roc_auc' if 'roc_auc' in scoring else 'accuracy'}_mean", ascending=False)
    print("Compact cross-validation comparison:")
    print(out.to_string(index=False))
    return out

def compact_holdout_regression(train, test, features, target, cat_cols, num_cols, log_target=False):
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42, min_samples_leaf=3),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }
    rows = []
    best = None
    best_mae = float("inf")
    y_train = np.log1p(train[target]) if log_target else train[target]
    for name, model in models.items():
        pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", model)])
        pipe.fit(train[features], y_train)
        pred = pipe.predict(test[features])
        pred = np.expm1(pred) if log_target else pred
        pred = np.maximum(0, pred)
        metrics = eval_regression(test[target], pred)
        rows.append({"Model": name, **metrics})
        if metrics["mae"] < best_mae:
            best_mae = metrics["mae"]
            best = (name, pipe)
    out = pd.DataFrame(rows).sort_values("mae")
    print("Time/holdout model comparison:")
    print(out.to_string(index=False))
    return out, best

def compact_randomized_tune_regressor(train, features, target, cat_cols, num_cols, log_target=False, n_iter=6):
    y_train = np.log1p(train[target]) if log_target else train[target]
    cv = KFold(n_splits=min(3, max(2, len(train) // 20)), shuffle=True, random_state=42)
    pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestRegressor(random_state=42))])
    search = RandomizedSearchCV(
        pipe,
        {
            "model__n_estimators": [100, 150, 250],
            "model__min_samples_leaf": [1, 3, 5, 8],
            "model__max_depth": [None, 3, 5, 8],
        },
        n_iter=n_iter,
        scoring="neg_mean_absolute_error",
        cv=cv,
        random_state=42,
        n_jobs=1,
    )
    search.fit(train[features], y_train)
    print("RandomizedSearchCV best params:", search.best_params_)
    print("RandomizedSearchCV best CV MAE:", float(-search.best_score_))
    return search.best_estimator_

def quick_eda(df, name, target_col=None, numeric_cols=None, categorical_cols=None):
    print(f"\\nEDA: {name}")
    print("Shape:", df.shape)
    safe_name = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    plot_dir = OUT_DIR / "eda-plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    missing = df.isna().mean().sort_values(ascending=False).head(10)
    print("\\nTop missing-value rates:")
    print(missing.to_string())
    if target_col and target_col in df.columns:
        print(f"\\nTarget distribution / summary: {target_col}")
        if pd.api.types.is_numeric_dtype(df[target_col]):
            print(df[target_col].describe().to_string())
            plt.figure(figsize=(7, 4))
            df[target_col].hist(bins=20)
            plt.title(f"{name}: {target_col} distribution")
            plt.xlabel(target_col)
            plt.ylabel("Count")
            plt.tight_layout()
        else:
            print(df[target_col].value_counts(dropna=False).head(20).to_string())
            plt.figure(figsize=(7, 4))
            df[target_col].value_counts(dropna=False).head(10).plot(kind="bar")
            plt.title(f"{name}: {target_col} distribution")
            plt.xlabel(target_col)
            plt.ylabel("Count")
            plt.tight_layout()
        plot_path = plot_dir / f"{safe_name}_{target_col}_distribution.png"
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close()
        print("Saved EDA plot:", plot_path)
    if numeric_cols:
        cols = [c for c in numeric_cols if c in df.columns]
        if cols:
            print("\\nNumeric feature summary:")
            print(df[cols].describe().T[["mean", "std", "min", "50%", "max"]].round(3).to_string())
    if categorical_cols:
        cols = [c for c in categorical_cols if c in df.columns]
        for col in cols[:5]:
            print(f"\\nTop values for {col}:")
            print(df[col].value_counts(dropna=False).head(10).to_string())

def time_split(df, date_col, test_frac=0.25):
    df = df.sort_values(date_col).copy()
    split_idx = max(1, int(len(df) * (1 - test_frac)))
    split_idx = min(split_idx, len(df) - 1)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def safe_classifier_split(X, y, test_size=0.25):
    stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
    return train_test_split(X, y, test_size=test_size, random_state=42, stratify=stratify)

def score_bands(scores):
    scores = pd.Series(scores).astype(float)
    if len(scores) == 0 or scores.nunique(dropna=True) < 2:
        return pd.Series(["Medium"] * len(scores), index=scores.index)
    labels = ["Low", "Medium", "High", "Very High"]
    q = min(4, scores.nunique(dropna=True), len(scores))
    try:
        return pd.qcut(scores.rank(method="first"), q=q, labels=labels[:q], duplicates="drop").astype(str)
    except Exception:
        return pd.Series(["Medium"] * len(scores), index=scores.index)

def prep(cat_cols, num_cols):
    return ColumnTransformer([("cat", make_encoder(), cat_cols), ("num", "passthrough", num_cols)])

def export_predictions_json(prediction_type, entity_type, df_out, id_col, score_col, label_col=None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    excluded = {id_col, score_col}
    if label_col:
        excluded.add(label_col)
    rows = []
    for _, row in df_out.iterrows():
        payload = {k: v for k, v in row.items() if k not in excluded}
        rows.append({
            "predictionType": prediction_type,
            "entityType": entity_type,
            "entityId": int(row[id_col]),
            "score": float(row[score_col]),
            "label": None if label_col is None or pd.isna(row[label_col]) else str(row[label_col]),
            "payloadJson": json.dumps(payload, default=str),
        })
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print(f"Wrote {len(rows)} predictions:", out_path)


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [ ]:
supporters = require_csv("supporters")
donations = require_csv("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
donations = donations.dropna(subset=["donation_date","supporter_id","channel_source"]).sort_values(["supporter_id","donation_date"]).copy()
donations["amount"] = numeric(donations["amount"])
donations["estimated_value"] = numeric(donations["estimated_value"])
donations["is_recurring_bool"] = as_bool(donations["is_recurring"])
donations["next_channel"] = donations.groupby("supporter_id")["channel_source"].shift(-1)
donations["next_date"] = donations.groupby("supporter_id")["donation_date"].shift(-1)
donations["days_to_next"] = (donations["next_date"] - donations["donation_date"]).dt.days
df = donations[(donations["next_channel"].notna()) & (donations["days_to_next"].between(1, 365))].copy()
df = df.merge(supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]], on="supporter_id", how="left")
df["donations_so_far"] = df.groupby("supporter_id").cumcount() + 1
df["prev_date"] = df.groupby("supporter_id")["donation_date"].shift(1)
df["recency_days"] = (df["donation_date"] - df["prev_date"]).dt.days.fillna(9999).clip(lower=0)
channel_values = ["Campaign", "Direct", "Event", "PartnerReferral", "SocialMedia"]
for channel in channel_values:
    current = df["channel_source"].eq(channel).astype(int)
    df[f"prior_{channel}"] = current.groupby(df["supporter_id"]).cumsum() - current
cat_cols = ["supporter_type","relationship_type","region","country","acquisition_channel","donation_type","campaign_name","channel_source","impact_unit"]
num_cols = ["amount","estimated_value","donations_so_far","recency_days","is_recurring_bool"] + [f"prior_{c}" for c in channel_values]
df[cat_cols] = df[cat_cols].fillna("Unknown")
fill_numeric_median(df, num_cols)
train, test = time_split(df, "donation_date")
print("Rows:", len(df), "Classes:", sorted(df["next_channel"].unique()))
quick_eda(df, "Next-channel modeling table", target_col="next_channel", numeric_cols=num_cols, categorical_cols=cat_cols)


## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [ ]:
features = cat_cols + num_cols
predictive = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=3))])
explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))])


## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [ ]:
print("Baseline:", classification_baseline(train["next_channel"], test["next_channel"]))
compact_cv_classification(train[features], train["next_channel"], cat_cols, num_cols, scoring_metric="accuracy")
predictive.fit(train[features], train["next_channel"])
pred = predictive.predict(test[features])
print("Predictive accuracy:", float(accuracy_score(test["next_channel"], pred)))
proba = predictive.predict_proba(test[features])
classes = predictive.named_steps["model"].classes_
print("Predictive top-2 accuracy:", float(top_k_accuracy_score(test["next_channel"], proba, k=2, labels=classes)))

explanatory.fit(train[features], train["next_channel"])
pred_exp = explanatory.predict(test[features])
print("Explanatory accuracy:", float(accuracy_score(test["next_channel"], pred_exp)))


## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [ ]:
print("Top predictive features:")
print(top_features(predictive).to_string(index=False))
print("Top explanatory relationships:")
print(top_features(explanatory).to_string(index=False))
print_business_takeaway("Treat the next-channel result as a ranked recommendation. Top-2 accuracy is more useful than exact-channel accuracy because outreach teams can test two good options.")


## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [ ]:
latest = donations.groupby("supporter_id").tail(1).copy()
latest = latest.merge(supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]], on="supporter_id", how="left")
history = donations.groupby("supporter_id").agg(
    donations_so_far=("donation_id","count"),
    prev_date=("donation_date", lambda s: s.iloc[-2] if len(s) >= 2 else pd.NaT),
).reset_index()
latest = latest.merge(history, on="supporter_id", how="left")
latest["recency_days"] = (latest["donation_date"] - latest["prev_date"]).dt.days.fillna(9999).clip(lower=0)
latest["is_recurring_bool"] = as_bool(latest["is_recurring"])
channel_counts = donations.pivot_table(index="supporter_id", columns="channel_source", values="donation_id", aggfunc="count", fill_value=0).reset_index()
latest = latest.merge(channel_counts, on="supporter_id", how="left")
for channel in channel_values:
    latest[f"prior_{channel}"] = latest[channel] if channel in latest.columns else 0
latest[cat_cols] = latest[cat_cols].fillna("Unknown")
fill_numeric_median(latest, num_cols)
proba = predictive.predict_proba(latest[features])
classes = predictive.named_steps["model"].classes_
best_idx = proba.argmax(axis=1)
latest["predicted_channel"] = [classes[i] for i in best_idx]
latest["confidence"] = proba.max(axis=1)
export_predictions_json(
    "next_channel_source",
    "Supporter",
    latest[["supporter_id","confidence","predicted_channel","channel_source","campaign_name","acquisition_channel","supporter_type"]],
    "supporter_id",
    "confidence",
    "predicted_channel",
)
